In [1]:
import numpy as np
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
import matplotlib as mpl

import multiprocessing as mp
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy import stats
import os
from stoch_sim_model import *
from joint_plot_grid_nb import *
import seaborn as sns

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'

In [2]:
# define global parameters
N_0 = 20
infection_type = "prim" #"prim", "sec"
sim_type = "agent" #"pop_ode", "agent"
reg = np.array([1.0, 1.0, -1.0, -1.0, 1.0, 1.0])
pnts = 160

virs = np.array(np.meshgrid(d_S*np.array([2, 20]), 
                            K_IE*np.array([1, 10]))).T.reshape(-1,2)

In [ ]:
# Plot trends for different networks based on signal correlation
n = len(virs)
n_d_I, n_K_IE = len(np.unique(virs[:,1])), len(np.unique(virs[:,0]))
fig, axs = plt.subplots(n_K_IE, n_d_I, figsize=(8, 6), sharex=True, sharey=True)

for l, (d_I, K_IE) in enumerate(virs):
    
    x,y = l % n_d_I, int(np.floor(l/n_K_IE))
    
    regs, dyn, ts, lin_comp, p_diff, Eff, mycNa, myccM, mycEin, mycEout = agent_stoch_sim(N_0 = N_0, b_I = b_I, K_IE = K_IE,
                                                                                      d_I = d_I,
                                                                                      regulation_coeffs =  reg,
                                                                                      infection = infection_type,
                                                                                      vir_model = "dep_harm",
                                                                                      reg_logs = np.array([1,1,1]))

    S, I, A, Na, E, cM, eM, H, I_d_I, I_d_S = dyn[:,-10], dyn[:,-9], dyn[:,-8], dyn[:,-7], dyn[:,-6], dyn[:,-5], dyn[:,-4], dyn[:,-3], dyn[:,-2], dyn[:,-1]
    
    keep = [i*int(len(E)/pnts) for i in np.arange(0,pnts)]
    sus = S[keep]
    naive = Na[keep]
    eff = E[keep]
    mem = (cM+eM)[keep]
    p_load = I[keep]/np.max(I)
    time = ts[keep]

    im = axs[x,y].scatter(time, naive, marker ='s', c =p_load, cmap='coolwarm', 
                label = r'Act. naive, $N^*$') #, norm=mpl.colors.LogNorm())
    axs[x,y].plot(time, naive, 'k--')
    axs[x,y].scatter(time, eff, marker ='o', c =p_load, cmap='coolwarm', 
                label = r'Effector, $E$')
    axs[x,y].plot(time, eff, 'k--')
    axs[x,y].scatter(time, mem, marker ='*', c =p_load, cmap='coolwarm',
                label = r'Memory, $cM+eM$')
    axs[x,y].plot(time, mem, 'k--')
    axs[x,y].set_title(r"$d_I =$ "+str(np.round(d_I,3))+r", $K_{IE} =$ "+str(np.round(K_IE,0)), fontsize = 8)
    #axs[x,y].set_title("d_I = {}".format(np.round(d_I,3))+", K_IE = {}".format(K_IE), fontsize = 8)
    
    if x == 0 and y == 0:
        axs[x,y].legend(fontsize = 8)

for i, ax in enumerate(axs.flat):
    ax.set(xlabel= r"time, $t$ (days)",
           ylabel= 'cellular response')
    ax.set_yscale('log')

for ax in axs.flat:
    ax.label_outer()
    
cb = fig.colorbar(im, ax=axs, orientation='vertical')
cb.set_label(r"Infection progression, $\frac{I}{max(I_{0 \to t})}$")
    
plt.savefig('_figs/dI_KIE_response_dynamics', dpi=300, bbox_inches='tight')

/mmfs1/home/oukogu/github/infoimmune/stoch_sim_model.py:489: RuntimeWarning: divide by zero encountered in double_scalars
  b_unbind_t[j] = np.fmin((K_IE/delta + np.sum(N_m[i-1] + Na_m[i-1] + cMa_m[i] + Ein_m[i]) + p_tcr[j]*Ain[i])/(p_tcr[j]*char_times[1]*Ain[i]), 1/(dt*trans_steps[1])) # set fastest rate of unbinding events
